# Dairy KML Generator Notebook
This notebook loads a CSV file of dairies and generates a KML file with layers for digesters and non-digesters, scaled by MilkCows.

In [ ]:
import pandas as pd
import numpy as np
import html, re

# Load CSV file
csv_path = 'CADD v2.0.0 merged data.csv'
df = pd.read_csv(csv_path)


In [ ]:
# Define MilkCows categories
bins = [0,500,1000,2000,3000,4000,5000,6000,7000,8000,9000,10000,np.inf]
labels = ['0-500','500-1000','1000-2000','2000-3000','3000-4000','4000-5000','5000-6000','6000-7000','7000-8000','8000-9000','9000-10000','10000+']
df['MilkCategory'] = pd.cut(df['MilkCows'], bins=bins, labels=labels, include_lowest=True)


In [ ]:
# Correct digester labeling (yellow = digester, red = no-digester)
def classify(val):
    if pd.isna(val): return 'no_digester'
    if isinstance(val,str) and val.strip()=='#N/A': return 'digester'
    return 'no_digester'
df['Digester'] = df['OperationalYear'].apply(classify)


In [ ]:
# Build KML
def clean_xml(text):
    if pd.isna(text): return ''
    text = str(text)
    text = re.sub(r'[\x00-\x1F]', '', text)
    return html.escape(text)

top = "<?xml version='1.0'?><kml xmlns='http://www.opengis.net/kml/2.2'><Document>"
style_red = "<Style id='red'><IconStyle><color>ff0000ff</color><Icon><href>http://maps.google.com/mapfiles/kml/paddle/red-circle.png</href></Icon></IconStyle></Style>"
style_yellow = "<Style id='yellow'><IconStyle><color>ff00ffff</color><Icon><href>http://maps.google.com/mapfiles/kml/paddle/ylw-circle.png</href></Icon></IconStyle></Style>"
milk_styles = ''
for lab in labels:
    milk_styles += f"<Style id='{lab}'><IconStyle><color>ff00ff00</color></IconStyle></Style>"

folders = ''
for dig in ['digester','no_digester']:
    folders += f'<Folder><name>{dig}</name>'
    sub = df[df['Digester']==dig]
    for lab in labels:
        folders += f'<Folder><name>{lab}</name>'
        sub2 = sub[sub['MilkCategory']==lab]
        for _,row in sub2.iterrows():
            lat,lon = row['Latitude'],row['Longitude']
            desc = clean_xml(row['FacilityName'])
            style = 'yellow' if dig=='digester' else 'red'
            folders += f"<Placemark><styleUrl>#{style}</styleUrl><description>{desc}</description><Point><coordinates>{lon},{lat},0</coordinates></Point></Placemark>"
        folders += '</Folder>'
    folders += '</Folder>'

final = top + style_red + style_yellow + milk_styles + folders + '</Document></kml>'
with open('CA_Dairies_scaled2.kml','w') as f: f.write(final)
print('KML written.')
